## <code>NTLM IR Experiment</code>

In [1]:

import numpy as np
import pyterrier as pt
import os
from collections import defaultdict
import pandas as pd
from tqdm import tqdm

In [2]:
# Set JAVA_HOME environment variable
java_home = r"C:\Program Files\Java\jdk-11"   # adjust your java JDK directory
os.environ["JAVA_HOME"] = java_home

# Verify that JAVA_HOME is set correctly
print("JAVA_HOME set to:", os.environ.get("JAVA_HOME"))

if not pt.started():
  pt.init()


JAVA_HOME set to: C:\Program Files\Java\jdk-11


C:\Users\dolla\AppData\Local\Temp\ipykernel_17028\153548638.py:8: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\dolla\AppData\Local\Temp\ipykernel_17028\153548638.py:9: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
# main Function

import os
import pyterrier as pt
from indexing import load_index, get_collection_statistics
from embedding_manager import BertEmbeddingManager1 #monoBertEmbeddingManager   #BertEmbeddingManager1 # don't import gpt2 and BERT model at the same time
from retrieval import retrieve_and_rank_documents, create_dirichlet_pipeline
import pandas as pd

def load_topics_and_qrels(topics_path, qrels_path):
    """Load topics and qrels from CSV files and prepare them for evaluation."""
    topics = pd.read_csv(topics_path, sep=',')
    qrels = pd.read_csv(qrels_path, sep=',')
    topics['qid'] = topics['qid'].astype(str)
    qrels['qid'] = qrels['qid'].astype(str)
    qrels['docno'] = qrels['docno'].astype(str)
    return topics, qrels

def evaluate_results(topics, qrels, reranked_results_df, dirichlet_pipeline):
    """Run evaluation metrics on reranked and Dirichlet results."""
    return pt.Experiment(
        [reranked_results_df, dirichlet_pipeline],
        topics, 
        qrels, 
        eval_metrics=["map", "P_10", "ndcg"], 
        names=["Dirichlet", "NTLM"], 
        filter_by_qrels=True, 
        filter_by_topics=True
    )

def main():

    # Initialize the model
    #embeddings = monoBertEmbeddingManager('castorini/monobert-large-msmarco')
    embeddings = BertEmbeddingManager1("bert-base-uncased")
    #embeddings = BertEmbeddingManager('bert-base-uncased')
    #embeddings = GptEmbeddingManager('gpt2')


    if not pt.started():
        pt.init()

    # Load index
    base_dir = os.path.dirname(os.path.abspath("__file__"))  
    #index_path = os.path.join(base_dir, "index", "AP_index")
    index_path = os.path.join(base_dir, "index", "DOTGOV_index")
    #index_path = os.path.join(base_dir, "index", "WSJ_index")
    #index_path = os.path.join(base_dir, "index", "trec-covid_index")
    #index_path = os.path.join(base_dir, "index", "webis-touche2020-v2_index")
    #index_path = os.path.join(base_dir, "index", "msmarco-document-trec-dl-2020_index")
    #index_path = os.path.join(base_dir, "index", "scifact_index")
    #index_path = os.path.join(base_dir, "index", "nfCorpus_index")


    index = load_index(index_path)
    get_collection_statistics(index)
  
    # Load topics and qrels
    #topics_path = os.path.join("topics", "Ap_query.csv")
    #qrels_path = os.path.join("qrels", "Ap_qrels.csv")
    topics_path = os.path.join("topics", "DOTGOV_query.csv")
    qrels_path = os.path.join("qrels", "DOTGOV_qrels.csv")
    #topics_path = os.path.join("topics", "WSJ_query.csv")
    #qrels_path = os.path.join("qrels", "WSJ_qrels.csv")
    #topics_path = os.path.join("topics", "trec-covid_query.csv")
    #qrels_path = os.path.join("qrels", "trec-covid_qrels.csv")
    #topics_path = os.path.join("topics", "webis-touche2020_v2_query.csv")
    #qrels_path = os.path.join("qrels", "webis-touche2020_v2_qrels.csv")

    #topics_path = os.path.join("topics", "msmarco_trec_dl_2020_query.csv")
    #qrels_path = os.path.join("qrels", "msmarco_trec_dl_2020_qrels.csv")

    #topics_path = os.path.join("topics", "beir_scifact_test_query.csv")
    #qrels_path = os.path.join("qrels", "beir_scifact_test_qrels.csv")
    
    #topics_path = os.path.join("topics", "beir_nfcorpus_dev_query.csv")
    #qrels_path = os.path.join("qrels", "beir_nfcorpus_dev_qrels.csv")

    topics, qrels = load_topics_and_qrels(topics_path, qrels_path)

    # Note: do not forget to adjust the mu parramter in the retrieval.py
    dirichlet_pipeline = create_dirichlet_pipeline(index, mu=350, num_results=100, metadata=['text'])  # metadata can be ['text'] or ['body']
    #dirichlet_results = dirichlet_pipeline.transform(topics)

    # reranked using NTLM results
    reranked_results_df = retrieve_and_rank_documents(
        topics,
        embeddings,
        index,
        alpha=0.4,
        dirichlet_weight=0,
        ntlm_weight=1,
       
    )

    # Evaluate results
    evaluation_results = evaluate_results(topics, qrels, reranked_results_df, dirichlet_pipeline)
    print(evaluation_results)

if __name__ == "__main__":
    main()

Index already exists. Loading existing index...
15:44:56.488 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 875.9 MiB of memory would be required.


C:\Users\dolla\AppData\Local\Temp\ipykernel_17028\3596367393.py:40: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


Number of documents: 1247753
Number of terms: 2899192
Number of postings: 277718803
Number of fields: 0
Number of tokens: 902632129
Field names: []
Positions:   false



TerrierRetr(DirichletLM): 100%|██████████| 50/50 [00:01<00:00, 39.91q/s]


        name       map   P_10      ndcg
0       NTLM  0.121588  0.210  0.271460
1  Dirichlet  0.118951  0.192  0.262832


: 

In [6]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel

class BertEmbeddingManager22:
    def __init__(self, model_name='bert-base-uncased'):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name)
        
    def get_contextualized_embeddings(self, text):
        """
        Get contextualized embeddings (final hidden states) for an input text.
        """
        # Tokenize the input text
        inputs = self.tokenizer(text, return_tensors="pt")
        
        # Pass the input through the model to get contextualized embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # The last hidden state is the contextualized embedding
        contextualized_embedding = outputs.last_hidden_state.squeeze(0)
        
        return contextualized_embedding.numpy()

# Example usage
embedding_manager = BertEmbeddingManager22()
text = "This is a sample sentence."
contextualized_embeddings = embedding_manager.get_contextualized_embeddings(text)

print("Contextualized embeddings shape:", contextualized_embeddings.shape)


Contextualized embeddings shape: (8, 768)
